# PlantVillage: data exploration
Prints dataset shapes, class counts, image-size stats, and sample images.
Kaggle: attach `emmarex/plantdisease` (or turn Internet On and the notebook downloads it).

In [ ]:
import os, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_ROOT = "/kaggle/input/plantdisease"
EXT = {".jpg", ".jpeg", ".png"}


def scan(root):
    rows, seen = [], set()
    for dp, dn, fn in os.walk(root):
        dn.sort()
        imgs = sorted(f for f in fn if Path(f).suffix.lower() in EXT)
        if not imgs:
            continue
        cls = os.path.basename(dp)
        if cls in seen:          # skip nested duplicate copy of the class folders
            continue
        seen.add(cls)
        rows += [(os.path.join(dp, f), cls) for f in imgs]
    return rows


rows = scan(DATA_ROOT)
if not rows:
    DATA_ROOT = "/kaggle/input"
    rows = scan(DATA_ROOT)
if not rows:
    print("Dataset not attached here -> downloading emmarex/plantdisease with kagglehub (needs Internet ON)...")
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    DATA_ROOT = kagglehub.dataset_download("emmarex/plantdisease")
    rows = scan(DATA_ROOT)
if not rows:
    raise RuntimeError("No images found. Attach emmarex/plantdisease via Add Input, or run kagglehub.login() and re-run.")

print("root used:", DATA_ROOT)
df = pd.DataFrame(rows, columns=["path", "label"])
df["crop"] = df.label.str.split("_").str[0]
classes = sorted(df.label.unique())
print("DataFrame shape (rows, cols):", df.shape)
print("number of classes:", len(classes))
print("number of crops:", df.crop.nunique(), sorted(df.crop.unique()))
df.head()

## Class distribution

In [ ]:
counts = df.label.value_counts()
print(counts.to_string())
print(f"\nlargest / smallest = {counts.max()} / {counts.min()} = {counts.max() / counts.min():.1f}x")

fig, ax = plt.subplots(figsize=(9, 5))
counts.sort_values().plot.barh(ax=ax, color="#4c9f70")
ax.set_xlabel("images"); ax.set_title("Images per class")
for i, v in enumerate(counts.sort_values().values):
    ax.text(v + 20, i, str(v), va="center", fontsize=8)
plt.tight_layout(); plt.show()

## Image shape and size statistics

In [ ]:
# one image: the raw array shape the model will see before resizing
img0 = Image.open(df.path.iloc[0])
a0 = np.array(img0.convert("RGB"))
print("example file :", df.path.iloc[0])
print("PIL size (W,H):", img0.size, "| mode:", img0.mode)
print("numpy shape (H,W,C):", a0.shape, "| dtype:", a0.dtype, "| min/max:", a0.min(), a0.max())

# size / mode statistics on a random sample (reading all 20k headers is slow-ish but fine; sample 1500)
sample = df.sample(min(1500, len(df)), random_state=SEED)
sizes, modes = Counter(), Counter()
for p in sample.path:
    with Image.open(p) as im:
        sizes[im.size] += 1
        modes[im.mode] += 1
print("\nimage sizes (W,H) in sample:", dict(sizes.most_common(5)))
print("color modes in sample:", dict(modes))

In [ ]:
# tensor shapes as the training pipeline will produce them
try:
    import torch
    from torchvision import transforms
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    batch = torch.stack([tf(Image.open(p).convert("RGB")) for p in df.path.sample(8, random_state=SEED)])
    print("one batch tensor shape (B, C, H, W):", tuple(batch.shape), "| dtype:", batch.dtype)
    print("pixel range after ToTensor:", float(batch.min()), "to", float(batch.max()))
except ImportError:
    print("torch/torchvision not installed; skipping tensor shape")
print("\nfull dataset as one array would be:", (len(df), 224, 224, 3), "(N, H, W, C) at 224x224")

## Sample images: one per class

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, c in zip(axes.ravel(), classes):
    p = df[df.label == c].sample(1, random_state=SEED).path.iloc[0]
    im = Image.open(p).convert("RGB")
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{c}\n{im.size[0]}x{im.size[1]}", fontsize=8)
for ax in axes.ravel()[len(classes):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## Random sample (16 images) with labels

In [ ]:
rs = df.sample(16, random_state=SEED + 1)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, r) in zip(axes.ravel(), rs.iterrows()):
    ax.imshow(Image.open(r.path).convert("RGB")); ax.axis("off")
    ax.set_title(r.label, fontsize=7)
plt.tight_layout(); plt.show()

## Several images from one class (change `CLASS` to look closer)

In [ ]:
CLASS = "Tomato_Late_blight"
if CLASS not in classes:
    CLASS = classes[0]
sub = df[df.label == CLASS].sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, p in zip(axes.ravel(), sub.path):
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
fig.suptitle(CLASS)
plt.tight_layout(); plt.show()